# EEG-Based Depression Screening from 8-Channel Resting-State EEG
by Hassan Ugail, Newton Howard, Ali Ahmed Elmahmudi and Zied Mnasri

## Subject-wise evaluation, compact baselines, connectivity ablation

This notebook consolidates the final experimental pipeline into one place.

## What this notebook does
It loads the processed outputs from the preprocessing notebook and reproduces the final experimental package:

1. **Correct subject identity handling**
   - uses `subject_key = label + "_" + subject_id`
   - avoids merging `H_S1` and `MDD_S1`

2. **Experiment 1 — Compact 8-channel baselines**
   - Extra Trees on asymmetry-aware spectral features
   - MLP on the same feature vector
   - compact 1D CNN on raw 8-channel segments

3. **Experiment 2 — Connectivity ablation**
   - Extra Trees on spectral features only
   - Extra Trees on connectivity features only
   - Extra Trees on spectral + connectivity fusion
   - MLP on spectral + connectivity fusion

4. **Publication-ready outputs**
   - manuscript tables
   - compact results CSV files
   - high-resolution figures with non-overlapping labels

## Dataset
This notebook assumes you already generated the processed files from the public dataset:

**Mumtaz, Wajid (2016). _MDD Patients and Healthy Controls EEG Data (New)._ figshare. Dataset.**  
DOI: `10.6084/m9.figshare.4244171.v2`

## Input assumption
This notebook expects the processed outputs from the earlier preprocessing stage under:

`/content/drive/MyDrive/.../run_subjectwise_baselines_v1/processed/`

Note: "run_subjectwise_baselines_v1/processed/" is available at:
https://drive.google.com/drive/folders/1K4J-jqfWtoG7njM21aOgc2Ct7QgYf4W9?usp=sharing

## Output philosophy
Only a small set of result files is written so analysis can be done elsewhere if needed.

In [ ]:
!pip install -q pandas numpy scipy scikit-learn matplotlib openpyxl torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports and configuration

In [ ]:
import os
import re
import json
import math
import random
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import coherence
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef,
    confusion_matrix, roc_auc_score, precision_score, recall_score
)
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
# =========================
# USER CONFIGURATION
# =========================

BASE_DIR = Path("/content/drive/MyDrive/.../")
INPUT_DIR = BASE_DIR / "run_subjectwise_baselines_v1" / "processed"

RUN_DIR = BASE_DIR / "run_final_public_notebook_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Output files only
FINAL_REPEAT_METRICS = RUN_DIR / "final_repeat_metrics.csv"
FINAL_SUMMARY = RUN_DIR / "final_summary.csv"
FINAL_SUBJECT_PREDICTIONS = RUN_DIR / "final_subject_predictions.csv"
FINAL_TOP_FEATURES = RUN_DIR / "final_top_features.csv"
FINAL_MANUSCRIPT_TABLE = RUN_DIR / "final_manuscript_table.csv"
FINAL_MANUSCRIPT_TABLE_XLSX = RUN_DIR / "final_manuscript_table.xlsx"
FIG_BASELINES = RUN_DIR / "fig_baselines_8ch.png"
FIG_FUSION = RUN_DIR / "fig_fusion_ablation.png"
FIG_IMPORTANCE = RUN_DIR / "fig_top_feature_importance.png"
MANIFEST_PATH = RUN_DIR / "manifest.json"

RANDOM_SEED = 42
N_REPEATS = 10
TEST_SIZE_SUBJECT_FRACTION = 0.20
VAL_SIZE_FROM_TRAIN_SUBJECTS = 0.20

BATCH_SIZE = 64
CNN_EPOCHS = 25
CNN_LR = 1e-3
EARLY_STOPPING_PATIENCE = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FULL_CHANNELS = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8",
    "T7", "C3", "Cz", "C4", "T8",
    "P7", "P3", "Pz", "P4", "P8", "O1", "O2"
]
CHANNELS_8CH = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4"]

CONNECTIVITY_BANDS = {
    "alpha": (8.0, 12.0),
    "beta": (12.0, 30.0),
}

## Helper functions

In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def specificity_from_cm(cm: np.ndarray) -> float:
    if cm.shape != (2, 2):
        return np.nan
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def safe_auc(y_true, y_prob) -> float:
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_prob)
    except Exception:
        return np.nan

def compute_metrics(y_true, y_pred, y_prob) -> dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    sens = recall_score(y_true, y_pred, zero_division=0)
    spec = specificity_from_cm(cm)
    prec = precision_score(y_true, y_pred, zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": prec,
        "recall_sensitivity": sens,
        "specificity": spec,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 and len(np.unique(y_pred)) > 1 else np.nan,
        "auroc": safe_auc(y_true, y_prob),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }

def aggregate_subject_predictions(df_pred: pd.DataFrame, subject_col: str = "subject_key") -> pd.DataFrame:
    agg = df_pred.groupby(subject_col).agg(
        y_true=("y_true", "first"),
        prob=("prob", "mean"),
        n_segments=("prob", "size"),
        label=("label", "first"),
        condition=("condition", lambda s: "|".join(sorted(pd.unique(s.astype(str))))),
        subject_id=("subject_id", "first"),
    ).reset_index()
    agg["y_pred"] = (agg["prob"] >= 0.5).astype(int)
    return agg

def stratified_subject_holdout(subject_df: pd.DataFrame, test_fraction=0.2, seed=42):
    rng = np.random.RandomState(seed)
    train_subjects, test_subjects = [], []
    for label_value in sorted(subject_df["y"].unique()):
        sub = subject_df[subject_df["y"] == label_value]["subject_key"].tolist()
        sub = list(sub)
        rng.shuffle(sub)
        n_test = max(1, int(round(len(sub) * test_fraction)))
        test_subjects.extend(sub[:n_test])
        train_subjects.extend(sub[n_test:])
    return sorted(train_subjects), sorted(test_subjects)

def make_subject_label_table(df: pd.DataFrame) -> pd.DataFrame:
    subj = df.groupby("subject_key").agg(
        y=("label", lambda s: 1 if s.iloc[0] == "MDD" else 0),
        label=("label", "first"),
        subject_id=("subject_id", "first")
    ).reset_index()
    return subj

def get_spectral_feature_columns(feature_table: pd.DataFrame, selected_channels: list) -> list:
    meta_cols = {"file_name", "subject_id", "subject_key", "label", "condition"}
    cols = [c for c in feature_table.columns if c not in meta_cols]
    base_cols = []
    asym_cols = []
    for c in cols:
        if "__ASYM_" in c:
            asym_cols.append(c)
            continue
        parts = c.split("__")
        if len(parts) != 2:
            continue
        prefix, ch = parts
        if ch not in selected_channels:
            continue
        if prefix.endswith("_logabs") or prefix.endswith("_rel"):
            base_cols.append(c)

    keep_asym = []
    for c in asym_cols:
        m = re.search(r"__ASYM_([A-Za-z0-9]+)_([A-Za-z0-9]+)$", c)
        if m and m.group(1) in selected_channels and m.group(2) in selected_channels:
            keep_asym.append(c)

    return sorted(base_cols + keep_asym)

def build_connectivity_features(X_raw_8ch, sfreq, channel_names, bands):
    edge_pairs = list(combinations(range(len(channel_names)), 2))
    edge_names = []
    rows = []
    for i, j in edge_pairs:
        for band_name in bands.keys():
            edge_names.append(f"coh_{band_name}__{channel_names[i]}__{channel_names[j]}")
    for seg in X_raw_8ch:
        feats = []
        for i, j in edge_pairs:
            x = seg[i]
            y = seg[j]
            f, cxy = coherence(x, y, fs=sfreq, nperseg=min(len(x), 512))
            for band_name, (fmin, fmax) in bands.items():
                mask = (f >= fmin) & (f < fmax)
                val = float(np.mean(cxy[mask])) if np.any(mask) else 0.0
                feats.append(val)
        rows.append(feats)
    return pd.DataFrame(rows, columns=edge_names)

def fit_predict_prob(model, X_train, y_train, X_test):
    fitted = clone(model)
    fitted.fit(X_train, y_train)
    if hasattr(fitted, "predict_proba"):
        prob = fitted.predict_proba(X_test)[:, 1]
    else:
        score = fitted.decision_function(X_test)
        prob = 1 / (1 + np.exp(-score))
    pred = (prob >= 0.5).astype(int)
    return fitted, prob, pred

class CompactEEGCNN(nn.Module):
    def __init__(self, n_channels: int, n_samples: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        x = self.net(x)
        x = self.head(x)
        return x.squeeze(1)

def fit_predict_cnn(
    X_train, y_train, groups_train, X_test, seed,
    val_fraction=0.2, batch_size=64, epochs=25, lr=1e-3, patience=5, device="cpu"
):
    set_all_seeds(seed)

    train_meta = pd.DataFrame({"subject_key": groups_train, "y": y_train}).drop_duplicates()
    gss = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
    tr_sub_idx, va_sub_idx = next(gss.split(train_meta[["y"]], train_meta["y"], groups=train_meta["subject_key"]))

    tr_subjects = set(train_meta.iloc[tr_sub_idx]["subject_key"])
    va_subjects = set(train_meta.iloc[va_sub_idx]["subject_key"])

    tr_mask = np.array([g in tr_subjects for g in groups_train])
    va_mask = np.array([g in va_subjects for g in groups_train])

    X_tr = X_train[tr_mask]
    y_tr = y_train[tr_mask]
    X_va = X_train[va_mask]
    y_va = y_train[va_mask]

    mean = X_tr.mean(axis=(0, 2), keepdims=True)
    std = X_tr.std(axis=(0, 2), keepdims=True) + 1e-6

    X_tr = (X_tr - mean) / std
    X_va = (X_va - mean) / std
    X_te = (X_test - mean) / std

    tr_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.float32))
    va_ds = TensorDataset(torch.tensor(X_va, dtype=torch.float32), torch.tensor(y_va, dtype=torch.float32))
    te_x = torch.tensor(X_te, dtype=torch.float32)

    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False)

    model = CompactEEGCNN(n_channels=X_train.shape[1], n_samples=X_train.shape[2]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    best_state = None
    best_val = np.inf
    bad_epochs = 0

    for _ in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            xb = xb.to(device); yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in va_loader:
                xb = xb.to(device); yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_losses.append(loss.item())
        val_loss = float(np.mean(val_losses)) if len(val_losses) else np.inf
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        logits = model(te_x.to(device)).cpu().numpy()
    prob = 1 / (1 + np.exp(-logits))
    pred = (prob >= 0.5).astype(int)
    return prob, pred

def make_public_label(model_name: str) -> str:
    mapping = {
        "extratrees_8ch_features": "Extra Trees (features)",
        "mlp_8ch_features": "MLP (features)",
        "cnn1d_8ch_raw": "1D CNN (raw 8ch)",
        "extratrees_spectral": "ET (spectral)",
        "extratrees_connectivity": "ET (connectivity)",
        "extratrees_fusion": "ET (fusion)",
        "mlp_fusion": "MLP (fusion)",
    }
    return mapping.get(model_name, model_name)

## Load processed data and create the corrected subject key

In [ ]:
feature_table = pd.read_csv(INPUT_DIR / "feature_table.csv")
X_segments = np.load(INPUT_DIR / "X_segments.npy")
y_segments = np.load(INPUT_DIR / "y.npy")
groups_segments = np.load(INPUT_DIR / "groups.npy", allow_pickle=True)
conditions_segments = np.load(INPUT_DIR / "conditions.npy", allow_pickle=True)

feature_table["subject_id"] = feature_table["subject_id"].astype(str)
feature_table["label"] = feature_table["label"].astype(str)
feature_table["condition"] = feature_table["condition"].astype(str)
feature_table["subject_key"] = feature_table["label"] + "_" + feature_table["subject_id"]

group_subject_ids = pd.Series(groups_segments.astype(str))
group_labels = pd.Series(np.where(y_segments == 1, "MDD", "H"))
group_subject_keys = (group_labels + "_" + group_subject_ids).values

subject_df = make_subject_label_table(feature_table)

print("feature_table:", feature_table.shape)
print("X_segments:", X_segments.shape)
print("Unique subject_key count:", len(subject_df))
print("Subjects by class:")
print(subject_df["label"].value_counts())

feature_table: (8267, 330)
X_segments: (8267, 19, 2048)
Unique subject_key count: 63
Subjects by class:
label
MDD    33
H      30
Name: count, dtype: int64


## Prepare feature matrices and raw 8-channel tensor

In [ ]:
channel_idx = [FULL_CHANNELS.index(ch) for ch in CHANNELS_8CH]
X_raw_8ch = X_segments[:, channel_idx, :]
sfreq = 256.0

spectral_cols = get_spectral_feature_columns(feature_table, CHANNELS_8CH)
X_spectral = feature_table[spectral_cols].copy()

X_connectivity = build_connectivity_features(
    X_raw_8ch=X_raw_8ch,
    sfreq=sfreq,
    channel_names=CHANNELS_8CH,
    bands=CONNECTIVITY_BANDS
)

X_fusion = pd.concat([X_spectral.reset_index(drop=True), X_connectivity.reset_index(drop=True)], axis=1)

y_all = feature_table["label"].map({"H": 0, "MDD": 1}).astype(int).values
subject_keys_all = feature_table["subject_key"].astype(str).values
subject_ids_all = feature_table["subject_id"].astype(str).values
conditions_all = feature_table["condition"].astype(str).values

print("Spectral features:", X_spectral.shape)
print("Connectivity features:", X_connectivity.shape)
print("Fusion features:", X_fusion.shape)
print("Raw 8ch tensor:", X_raw_8ch.shape)

Spectral features: (8267, 100)
Connectivity features: (8267, 56)
Fusion features: (8267, 156)
Raw 8ch tensor: (8267, 8, 2048)


## Define the experimental models

In [ ]:
compact_models = {
    "extratrees_8ch_features": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),
    "mlp_8ch_features": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            alpha=1e-4,
            batch_size=64,
            learning_rate_init=1e-3,
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=RANDOM_SEED
        ))
    ]),
}

fusion_models = {
    "extratrees_spectral": {
        "X": X_spectral,
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", ExtraTreesClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=-1
            ))
        ])
    },
    "extratrees_connectivity": {
        "X": X_connectivity,
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", ExtraTreesClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=-1
            ))
        ])
    },
    "extratrees_fusion": {
        "X": X_fusion,
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", ExtraTreesClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=-1
            ))
        ])
    },
    "mlp_fusion": {
        "X": X_fusion,
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                alpha=1e-4,
                batch_size=64,
                learning_rate_init=1e-3,
                max_iter=200,
                early_stopping=True,
                validation_fraction=0.15,
                random_state=RANDOM_SEED
            ))
        ])
    },
}

## Run all experiments under repeated subject-wise holdout

In [ ]:
all_metric_rows = []
all_subject_pred_rows = []
feature_importance_rows = []

for repeat_idx in range(N_REPEATS):
    seed = RANDOM_SEED + repeat_idx
    set_all_seeds(seed)

    train_subjects, test_subjects = stratified_subject_holdout(
        subject_df,
        test_fraction=TEST_SIZE_SUBJECT_FRACTION,
        seed=seed
    )

    train_mask = np.isin(subject_keys_all, train_subjects)
    test_mask = np.isin(subject_keys_all, test_subjects)

    y_train = y_all[train_mask]
    y_test = y_all[test_mask]
    g_train = subject_keys_all[train_mask]
    g_test = subject_keys_all[test_mask]
    sid_test = subject_ids_all[test_mask]
    cond_test = conditions_all[test_mask]
    label_test = np.where(y_test == 1, "MDD", "H")

    print(f"===== Repeat {repeat_idx+1}/{N_REPEATS} | seed={seed} =====")
    tmp = pd.DataFrame({"subject_key": train_subjects + test_subjects, "split": ["train"]*len(train_subjects)+["test"]*len(test_subjects)})
    tmp = tmp.merge(subject_df[["subject_key", "label"]], on="subject_key", how="left")
    print(tmp.groupby(["split", "label"]).size())

    # Experiment 1: compact baselines
    Xf_train = X_spectral.loc[train_mask].reset_index(drop=True)
    Xf_test = X_spectral.loc[test_mask].reset_index(drop=True)

    for model_name, model in compact_models.items():
        fitted, prob, pred = fit_predict_prob(model, Xf_train, y_train, Xf_test)
        seg_df = pd.DataFrame({
            "repeat": repeat_idx, "seed": seed, "model": model_name,
            "subject_key": g_test, "subject_id": sid_test, "label": label_test,
            "condition": cond_test, "y_true": y_test, "prob": prob, "y_pred": pred,
        })
        subj_df = aggregate_subject_predictions(seg_df, subject_col="subject_key")
        subj_df["repeat"] = repeat_idx
        subj_df["seed"] = seed
        subj_df["model"] = model_name
        all_subject_pred_rows.append(subj_df)

        metrics = compute_metrics(subj_df["y_true"].values, subj_df["y_pred"].values, subj_df["prob"].values)
        metrics.update({"repeat": repeat_idx, "seed": seed, "model": model_name, "experiment": "compact_baselines", "n_subjects": len(subj_df), "n_segments": len(seg_df)})
        all_metric_rows.append(metrics)

        if model_name == "extratrees_8ch_features":
            clf = fitted.named_steps["clf"]
            feat_names = list(Xf_train.columns)
            importances = clf.feature_importances_
            top_idx = np.argsort(importances)[::-1][:15]
            for rank, idx in enumerate(top_idx, start=1):
                feature_importance_rows.append({
                    "repeat": repeat_idx, "model": model_name, "rank": rank,
                    "feature": feat_names[idx], "importance": float(importances[idx]),
                    "family": "spectral",
                })

    # Raw 1D CNN
    Xr_train = X_raw_8ch[train_mask]
    Xr_test = X_raw_8ch[test_mask]
    prob_cnn, pred_cnn = fit_predict_cnn(
        X_train=Xr_train, y_train=y_train, groups_train=g_train, X_test=Xr_test,
        seed=seed, val_fraction=VAL_SIZE_FROM_TRAIN_SUBJECTS, batch_size=BATCH_SIZE,
        epochs=CNN_EPOCHS, lr=CNN_LR, patience=EARLY_STOPPING_PATIENCE, device=DEVICE,
    )
    seg_df = pd.DataFrame({
        "repeat": repeat_idx, "seed": seed, "model": "cnn1d_8ch_raw",
        "subject_key": g_test, "subject_id": sid_test, "label": label_test,
        "condition": cond_test, "y_true": y_test, "prob": prob_cnn, "y_pred": pred_cnn,
    })
    subj_df = aggregate_subject_predictions(seg_df, subject_col="subject_key")
    subj_df["repeat"] = repeat_idx
    subj_df["seed"] = seed
    subj_df["model"] = "cnn1d_8ch_raw"
    all_subject_pred_rows.append(subj_df)
    metrics = compute_metrics(subj_df["y_true"].values, subj_df["y_pred"].values, subj_df["prob"].values)
    metrics.update({"repeat": repeat_idx, "seed": seed, "model": "cnn1d_8ch_raw", "experiment": "compact_baselines", "n_subjects": len(subj_df), "n_segments": len(seg_df)})
    all_metric_rows.append(metrics)

    # Experiment 2: connectivity ablation
    for model_name, spec in fusion_models.items():
        X_df = spec["X"]
        model = spec["model"]
        X_train = X_df.loc[train_mask].reset_index(drop=True)
        X_test = X_df.loc[test_mask].reset_index(drop=True)

        fitted, prob, pred = fit_predict_prob(model, X_train, y_train, X_test)
        seg_df = pd.DataFrame({
            "repeat": repeat_idx, "seed": seed, "model": model_name,
            "subject_key": g_test, "subject_id": sid_test, "label": label_test,
            "condition": cond_test, "y_true": y_test, "prob": prob, "y_pred": pred,
        })
        subj_df = aggregate_subject_predictions(seg_df, subject_col="subject_key")
        subj_df["repeat"] = repeat_idx
        subj_df["seed"] = seed
        subj_df["model"] = model_name
        all_subject_pred_rows.append(subj_df)

        metrics = compute_metrics(subj_df["y_true"].values, subj_df["y_pred"].values, subj_df["prob"].values)
        metrics.update({"repeat": repeat_idx, "seed": seed, "model": model_name, "experiment": "fusion_ablation", "n_subjects": len(subj_df), "n_segments": len(seg_df)})
        all_metric_rows.append(metrics)

        if model_name.startswith("extratrees"):
            clf = fitted.named_steps["clf"]
            feat_names = list(X_df.columns)
            importances = clf.feature_importances_
            top_idx = np.argsort(importances)[::-1][:15]
            family = "spectral" if model_name == "extratrees_spectral" else ("connectivity" if model_name == "extratrees_connectivity" else "fusion")
            for rank, idx in enumerate(top_idx, start=1):
                feature_importance_rows.append({
                    "repeat": repeat_idx, "model": model_name, "rank": rank,
                    "feature": feat_names[idx], "importance": float(importances[idx]),
                    "family": family,
                })

===== Repeat 1/10 | seed=42 =====
split  label
test   H         6
       MDD       7
train  H        24
       MDD      26
dtype: int64
===== Repeat 2/10 | seed=43 =====
split  label
test   H         6
       MDD       7
train  H        24
       MDD      26
dtype: int64
===== Repeat 3/10 | seed=44 =====
split  label
test   H         6
       MDD       7
train  H        24
       MDD      26
dtype: int64
===== Repeat 4/10 | seed=45 =====
split  label
test   H         6
       MDD       7
train  H        24
       MDD      26
dtype: int64
===== Repeat 5/10 | seed=46 =====
split  label
test   H         6
       MDD       7
train  H        24
       MDD      26
dtype: int64
===== Repeat 6/10 | seed=47 =====
split  label
test   H         6
       MDD       7
train  H        24
       MDD      26
dtype: int64
===== Repeat 7/10 | seed=48 =====
split  label
test   H         6
       MDD       7
train  H        24
       MDD      26
dtype: int64
===== Repeat 8/10 | seed=49 =====
split  label
t

## Save tables

In [ ]:
repeat_metrics_df = pd.DataFrame(all_metric_rows)
subject_predictions_df = pd.concat(all_subject_pred_rows, axis=0, ignore_index=True) if all_subject_pred_rows else pd.DataFrame()
top_features_df = pd.DataFrame(feature_importance_rows)

summary_df = (
    repeat_metrics_df
    .groupby(["experiment", "model"], as_index=False)
    .agg(
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        auroc_mean=("auroc", "mean"),
        auroc_std=("auroc", "std"),
        sensitivity_mean=("recall_sensitivity", "mean"),
        specificity_mean=("specificity", "mean"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        mcc_mean=("mcc", "mean"),
        n_subjects_mean=("n_subjects", "mean"),
    )
    .sort_values(["experiment", "balanced_accuracy_mean", "auroc_mean"], ascending=[True, False, False])
    .reset_index(drop=True)
)
summary_df["public_label"] = summary_df["model"].map(make_public_label)

if len(top_features_df):
    top_features_summary = (
        top_features_df
        .groupby(["model", "family", "feature"], as_index=False)
        .agg(
            importance_mean=("importance", "mean"),
            importance_count=("importance", "size"),
        )
        .sort_values(["model", "importance_mean"], ascending=[True, False])
        .groupby("model", as_index=False)
        .head(12)
        .reset_index(drop=True)
    )
else:
    top_features_summary = pd.DataFrame()

manuscript_table = summary_df[[
    "experiment", "public_label", "balanced_accuracy_mean", "balanced_accuracy_std",
    "auroc_mean", "auroc_std", "sensitivity_mean", "specificity_mean",
    "precision_mean", "f1_mean", "mcc_mean"
]].copy()

repeat_metrics_df.to_csv(FINAL_REPEAT_METRICS, index=False)
summary_df.to_csv(FINAL_SUMMARY, index=False)
subject_predictions_df.to_csv(FINAL_SUBJECT_PREDICTIONS, index=False)
top_features_summary.to_csv(FINAL_TOP_FEATURES, index=False)
manuscript_table.to_csv(FINAL_MANUSCRIPT_TABLE, index=False)
manuscript_table.to_excel(FINAL_MANUSCRIPT_TABLE_XLSX, index=False)

print("Saved:")
print(FINAL_REPEAT_METRICS)
print(FINAL_SUMMARY)
print(FINAL_SUBJECT_PREDICTIONS)
print(FINAL_TOP_FEATURES)
print(FINAL_MANUSCRIPT_TABLE)
print(FINAL_MANUSCRIPT_TABLE_XLSX)
display(manuscript_table)

## Figure 1 — Compact 8-channel baselines

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.6), constrained_layout=True)

df_plot = summary_df[summary_df["experiment"] == "compact_baselines"].copy()
order = ["extratrees_8ch_features", "mlp_8ch_features", "cnn1d_8ch_raw"]
df_plot["order"] = df_plot["model"].map({m:i for i,m in enumerate(order)})
df_plot = df_plot.sort_values("order")

labels = [make_public_label(m) for m in df_plot["model"]]
means = df_plot["balanced_accuracy_mean"].values
stds = df_plot["balanced_accuracy_std"].values

ax.bar(np.arange(len(labels)), means, width=0.62)
ax.errorbar(np.arange(len(labels)), means, yerr=stds, fmt="none", capsize=4, linewidth=1.5)
ax.set_ylabel("Subject-level balanced accuracy")
ax.set_title("Compact 8-channel baselines")
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylim(0.0, 1.05)

for i, v in enumerate(means):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", va="bottom", fontsize=11)

fig.savefig(FIG_BASELINES, bbox_inches="tight")
plt.show()
print(FIG_BASELINES)

## Figure 2 — Spectral vs connectivity vs fusion

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 4.8), constrained_layout=True)

df_plot = summary_df[summary_df["experiment"] == "fusion_ablation"].copy()
order = ["extratrees_spectral", "extratrees_connectivity", "extratrees_fusion", "mlp_fusion"]
df_plot["order"] = df_plot["model"].map({m:i for i,m in enumerate(order)})
df_plot = df_plot.sort_values("order")

labels = [make_public_label(m) for m in df_plot["model"]]
means = df_plot["balanced_accuracy_mean"].values
stds = df_plot["balanced_accuracy_std"].values

ax.bar(np.arange(len(labels)), means, width=0.62)
ax.errorbar(np.arange(len(labels)), means, yerr=stds, fmt="none", capsize=4, linewidth=1.5)
ax.set_ylabel("Subject-level balanced accuracy")
ax.set_title("Spectral vs connectivity vs fusion")
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylim(0.0, 1.05)

for i, v in enumerate(means):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", va="bottom", fontsize=11)

fig.savefig(FIG_FUSION, bbox_inches="tight")
plt.show()
print(FIG_FUSION)

## Figure 3 — Top feature importance preview

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)

# Spectral winner
spec_df = top_features_summary[top_features_summary["model"] == "extratrees_8ch_features"].head(10).copy()
spec_df = spec_df.sort_values("importance_mean", ascending=True)

axs[0].barh(spec_df["feature"], spec_df["importance_mean"])
axs[0].set_title("(a) Top spectral/asymmetry features")
axs[0].set_xlabel("Mean importance")

# Connectivity winner
conn_df = top_features_summary[top_features_summary["model"] == "extratrees_connectivity"].head(10).copy()
conn_df = conn_df.sort_values("importance_mean", ascending=True)

axs[1].barh(conn_df["feature"], conn_df["importance_mean"])
axs[1].set_title("(b) Top connectivity features")
axs[1].set_xlabel("Mean importance")

fig.savefig(FIG_IMPORTANCE, bbox_inches="tight")
plt.show()
print(FIG_IMPORTANCE)

## Compact narrative summary for the paper

In [ ]:
best_compact = summary_df[summary_df["experiment"] == "compact_baselines"].sort_values(
    ["balanced_accuracy_mean", "auroc_mean"], ascending=[False, False]
).head(1)

best_fusion = summary_df[summary_df["experiment"] == "fusion_ablation"].sort_values(
    ["balanced_accuracy_mean", "auroc_mean"], ascending=[False, False]
).head(1)

print("Best compact-baseline result")
display(best_compact[["public_label", "balanced_accuracy_mean", "balanced_accuracy_std", "auroc_mean", "auroc_std"]])

print("Best fusion-ablation result")
display(best_fusion[["public_label", "balanced_accuracy_mean", "balanced_accuracy_std", "auroc_mean", "auroc_std"]])

print("\nInterpretation guide:")
print("- If Extra Trees (features) remains best, the manuscript anchor stays the spectral/asymmetry baseline.")
print("- If ET (connectivity) is competitive, discuss complementary network-level information.")
print("- If ET (fusion) does not exceed the spectral baseline, state clearly that naive early fusion did not improve performance.")

Best compact-baseline result


,public_label,balanced_accuracy_mean,balanced_accuracy_std,auroc_mean,auroc_std
0,Extra Trees (features),0.934524,0.063801,0.985714,0.037562


Best fusion-ablation result


,public_label,balanced_accuracy_mean,balanced_accuracy_std,auroc_mean,auroc_std
3,ET (fusion),0.934524,0.050273,0.985714,0.037562



Interpretation guide:
- If Extra Trees (features) remains best, the manuscript anchor stays the spectral/asymmetry baseline.
- If ET (connectivity) is competitive, discuss complementary network-level information.
- If ET (fusion) does not exceed the spectral baseline, state clearly that naive early fusion did not improve performance.


## Manifest

In [ ]:
manifest = {
    "input_dir": str(INPUT_DIR),
    "run_dir": str(RUN_DIR),
    "subject_key_fix": "subject_key = label + '_' + subject_id",
    "n_repeats": N_REPEATS,
    "channels_8ch": CHANNELS_8CH,
    "connectivity_bands": CONNECTIVITY_BANDS,
    "saved_outputs": {
        "final_repeat_metrics_csv": str(FINAL_REPEAT_METRICS),
        "final_summary_csv": str(FINAL_SUMMARY),
        "final_subject_predictions_csv": str(FINAL_SUBJECT_PREDICTIONS),
        "final_top_features_csv": str(FINAL_TOP_FEATURES),
        "final_manuscript_table_csv": str(FINAL_MANUSCRIPT_TABLE),
        "final_manuscript_table_xlsx": str(FINAL_MANUSCRIPT_TABLE_XLSX),
        "fig_baselines_png": str(FIG_BASELINES),
        "fig_fusion_png": str(FIG_FUSION),
        "fig_importance_png": str(FIG_IMPORTANCE),
    }
}
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved manifest:", MANIFEST_PATH)